In [ ]:
#when using pydantic parsing format instrcution within the prompt template , it does not gurantee that the output will be structured
#it may or may not be in json format , hence we use with_structured_output to ensure the output is structured
#so with strcutured output is highly reliable , while pydantic parsing in react llms is prompt dependent 

In [1]:
from dotenv import load_dotenv
import os 
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from langchain.tools import tool
from pydantic import BaseModel , Field 
from langchain_classic.agents.react.agent import create_react_agent
from langchain_classic.agents import AgentExecutor

from langchain_classic import hub #Think of it like GitHub, but specifically for LangChain prompts and chains.
#so this is a place where you can find structured prompts and chains 

load_dotenv(override=True)

api_key = os.getenv("LLM_API_KEY")
if not api_key:
    print("LLM_API_KEY not found")
else:
    print("Api key exists")

Api key exists


In [2]:
from typing import List
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

In [3]:
class Source(BaseModel):
    """Schema for a source used by the agent"""

    url : str = Field(description="The url of the source")

class AgentResponse(BaseModel):
    """Scehama for agent response with answer and sources"""

    answer : str = Field(description="Agent's answer to query")
    sources : List[Source] = Field(
        default_factory=list , description="List of sources used by the agent to answer the query"
    )


In [4]:
react_prompt_with_format_instructions = """
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}

IMPORTANT:
You must respond with exactly ONE of the following:
- An Action and Action Input
- OR a Final Answer
Never both in the same response.

"""



In [ ]:
react_prompt_with_format_instructions = PromptTemplate(
    template = react_prompt_with_format_instructions,
    input_variables = ["input" , "agent_scratchpad" , "tools" , "tool_names" , "format_instructions"]
)

In [6]:
tools = [TavilySearch()]
llm = ChatOpenAI(model="openai/gpt-4o", temperature=0 , api_key=os.getenv("LLM_API_KEY") , base_url=os.getenv("BASE_URL"))
structured_llm = llm.with_structured_output(AgentResponse)
agent = create_react_agent(
    llm, 
    tools,  
    react_prompt_with_format_instructions,
)

agent_executor = AgentExecutor(agent = agent , tools = tools , verbose = True )
extract_output = RunnableLambda(lambda x : x["output"])


In [7]:
chain = agent_executor | extract_output | structured_llm

result = chain.invoke(
    input={
        "input":"search for 3 Ai jobs in india"
    }
) 



> Entering new AgentExecutor chain...
Action: tavily_search
Action Input: "AI jobs in India 2023"{'query': 'AI jobs in India 2023', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://skillfloor.com/blog/artificial-intelligence-jobs-in-india', 'title': 'Artificial Intelligence Jobs in India - Skillfloor', 'content': 'The job market for AI is rapidly expanding in India, offering exciting opportunities in sectors such as technology, healthcare, and finance. The', 'score': 0.9998776, 'raw_content': None}, {'url': 'https://opencv.org/blog/from-aspiring-to-employed-a-step-by-step-guide-to-ai-jobs-in-india/', 'title': 'From Aspiring to Employed: A Step-by-Step Guide to AI Jobs in India', 'content': 'Job seekers in the AI sector have many options with dedicated job portals like Naukri, LinkedIn, and Indeed, providing ample opportunities to explore and', 'score': 0.99973875, 'raw_content': None}, {'url': 'https://wellfound.com/role/l/ai-engineer/india', 't

In [8]:
result 

AgentResponse(answer='### 1. Data Scientist\n**Role Overview:**\n- **Responsibilities:**\n  - Analyze large datasets to extract actionable insights.\n  - Develop predictive models and algorithms to solve complex business problems.\n  - Collaborate with cross-functional teams to integrate data-driven solutions.\n- **Skills Required:**\n  - Proficiency in statistical analysis and data mining.\n  - Expertise in programming languages like Python or R.\n  - Strong understanding of machine learning techniques and tools.\n- **Industries Hiring:**\n  - Technology, E-commerce, Healthcare, Finance.\n\n### 2. Python ML Developer\n**Role Overview:**\n- **Responsibilities:**\n  - Design and implement machine learning models using Python.\n  - Optimize and scale machine learning algorithms for production.\n  - Work closely with data engineers to ensure seamless data flow.\n- **Skills Required:**\n  - Strong programming skills in Python.\n  - Experience with libraries such as TensorFlow, PyTorch, or 

In [9]:
# with_structured_output() wraps an LLM so that its response
# MUST follow a predefined schema (Pydantic / TypedDict).

# Instead of returning free-form text,
# the LLM returns a validated, typed Python object.

# ----------------------------
# WITHOUT structured output
# ----------------------------

# llm.invoke("Give me a user profile")
# Output: "The user's name is Zaid, he is 25 years old and lives in India."
# Problems:
# - Free-form text
# - Hard to parse
# - Format may change anytime

# ----------------------------
# WITH structured output
# ----------------------------

# 1. Define a schema using Pydantic
# class UserProfile(BaseModel):
#     name: str
#     age: int
#     country: str

# 2. Wrap the LLM
# structured_llm = llm.with_structured_output(UserProfile)

# 3. Invoke the model
# result = structured_llm.invoke("Give me a user profile")

# Result is NOT text, but a Python object:
# UserProfile(name="Zaid", age=25, country="India")

# ----------------------------
# What with_structured_output() does internally
# ----------------------------

# 1. Automatically modifies the prompt to force JSON output
#    matching the provided schema.

# 2. Uses model-native structured output features when available:
#    - OpenAI: JSON mode / function calling
#    - Anthropic: structured response formats
#    - Others: constrained JSON prompting

# 3. Parses and validates the model output:
#    - Converts JSON → Python object
#    - Raises an error if schema validation fails
#    - Does NOT silently accept invalid output

# ----------------------------
# What it does NOT do
# ----------------------------

# - Does NOT improve reasoning quality
# - Does NOT control chain-of-thought
# - Does NOT work inside ReAct agent loops
# - Does NOT guarantee correctness of values (only structure)

# ----------------------------
# Why it does NOT work with ReAct agents
# ----------------------------

# ReAct agents generate:
# Thought → Action → Observation → Thought → Final Answer
# This flow cannot be represented as a single JSON object.

# Therefore:
# ❌ Do NOT wrap the LLM inside create_react_agent() with structured output
# ✅ Use structured output ONLY after the agent finishes reasoning

# ----------------------------
# Correct usage with ReAct
# ----------------------------

# raw_output = agent_executor.invoke({"input": question})
# final_output = llm.with_structured_output(MySchema).invoke(
#     f"Convert this into structured output:\n{raw_output['output']}"
# )

# ----------------------------
# When to use with_structured_output()
# ----------------------------

# - API responses
# - Database writes
# - Tool inputs
# - UI payloads
# - Evaluation results
# - Any output consumed by code

# ----------------------------
# One-line takeaway
# ----------------------------

# with_structured_output() turns an LLM
# from a text generator into a schema-enforced, type-safe function.


In [10]:
from pprint import pprint

pprint(result.model_dump())


{'answer': '### 1. Data Scientist\n'
           '**Role Overview:**\n'
           '- **Responsibilities:**\n'
           '  - Analyze large datasets to extract actionable insights.\n'
           '  - Develop predictive models and algorithms to solve complex '
           'business problems.\n'
           '  - Collaborate with cross-functional teams to integrate '
           'data-driven solutions.\n'
           '- **Skills Required:**\n'
           '  - Proficiency in statistical analysis and data mining.\n'
           '  - Expertise in programming languages like Python or R.\n'
           '  - Strong understanding of machine learning techniques and '
           'tools.\n'
           '- **Industries Hiring:**\n'
           '  - Technology, E-commerce, Healthcare, Finance.\n'
           '\n'
           '### 2. Python ML Developer\n'
           '**Role Overview:**\n'
           '- **Responsibilities:**\n'
           '  - Design and implement machine learning models using Python.\n'
      

In [11]:
print(result.model_dump_json(indent=2))


{
  "answer": "### 1. Data Scientist\n**Role Overview:**\n- **Responsibilities:**\n  - Analyze large datasets to extract actionable insights.\n  - Develop predictive models and algorithms to solve complex business problems.\n  - Collaborate with cross-functional teams to integrate data-driven solutions.\n- **Skills Required:**\n  - Proficiency in statistical analysis and data mining.\n  - Expertise in programming languages like Python or R.\n  - Strong understanding of machine learning techniques and tools.\n- **Industries Hiring:**\n  - Technology, E-commerce, Healthcare, Finance.\n\n### 2. Python ML Developer\n**Role Overview:**\n- **Responsibilities:**\n  - Design and implement machine learning models using Python.\n  - Optimize and scale machine learning algorithms for production.\n  - Work closely with data engineers to ensure seamless data flow.\n- **Skills Required:**\n  - Strong programming skills in Python.\n  - Experience with libraries such as TensorFlow, PyTorch, or scikit-

In [12]:
from IPython.display import Markdown, display

display(Markdown(result.answer))


### 1. Data Scientist
**Role Overview:**
- **Responsibilities:**
  - Analyze large datasets to extract actionable insights.
  - Develop predictive models and algorithms to solve complex business problems.
  - Collaborate with cross-functional teams to integrate data-driven solutions.
- **Skills Required:**
  - Proficiency in statistical analysis and data mining.
  - Expertise in programming languages like Python or R.
  - Strong understanding of machine learning techniques and tools.
- **Industries Hiring:**
  - Technology, E-commerce, Healthcare, Finance.

### 2. Python ML Developer
**Role Overview:**
- **Responsibilities:**
  - Design and implement machine learning models using Python.
  - Optimize and scale machine learning algorithms for production.
  - Work closely with data engineers to ensure seamless data flow.
- **Skills Required:**
  - Strong programming skills in Python.
  - Experience with libraries such as TensorFlow, PyTorch, or scikit-learn.
  - Understanding of data structures and algorithms.
- **Industries Hiring:**
  - IT Services, Startups, Research Labs, Automotive.

### 3. Artificial Intelligence Specialist
**Role Overview:**
- **Responsibilities:**
  - Develop AI-driven solutions to enhance business processes.
  - Research and implement new AI technologies and methodologies.
  - Provide technical leadership and guidance on AI projects.
- **Skills Required:**
  - Deep knowledge of AI frameworks and platforms.
  - Experience in natural language processing and computer vision.
  - Strong problem-solving and analytical skills.
- **Industries Hiring:**
  - Telecommunications, Manufacturing, Education, Government.

### Opportunities and Growth
- **Expanding Market:** The AI job market in India is growing rapidly, driven by the increasing adoption of AI technologies across various sectors.
- **Diverse Applications:** AI professionals can work on projects ranging from healthcare diagnostics to financial forecasting and autonomous vehicles.
- **Career Advancement:** With continuous learning and skill enhancement, AI professionals can advance to roles such as AI Architect, Chief Data Officer, or Head of AI.

### Conclusion
The AI job market in India offers a plethora of opportunities for skilled professionals. With the right expertise and a passion for innovation, individuals can make significant contributions to the AI landscape and drive technological advancements across industries.